## Busca de Hiperparâmetros

In [ ]:

import os
import json
import time
import random
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import optuna
from optuna.pruners import MedianPruner

import tensorflow as tf

# =========================
# CONFIG
# =========================
CSV_PATH = "bd_EstacaoVargemFria_e_Pesca.csv"
STATION_NAME = "Estação Pesca - UFRPE"
TIME_COL = "Data estação"

CTX = 7
KERNEL_SIZE = 5

MAX_EPOCHS = 100
PATIENCE = 15
VAL_TAIL_FRAC = 0.10
SEED = 42

MAX_PARAMS_BUDGET = 12000  

N_TRIALS = 80
TOP_K = 12  

OUT_DIR = "hpo_tinycnn_tflite_out"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# UTIL
# =========================
def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def load_dataframe():
    df = pd.read_csv(CSV_PATH, on_bad_lines="skip")
    df = df[df["Nome"].astype(str).str.strip() == STATION_NAME].copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])

    cols = ["Temperatura", "Umidade", "Velocidade Vento", "Rajada Vento", "Precipitação dia"]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.sort_values(TIME_COL)
    df = df.infer_objects(copy=False).interpolate().dropna().reset_index(drop=True)
    return df

def build_sequences(df):
    cols_X = ["Temperatura", "Umidade", "Velocidade Vento", "Rajada Vento", "Precipitação dia"]
    arr = df[cols_X].values.astype(np.float32)
    y_all = df["Precipitação dia"].values.astype(np.float32)

    X_list, y_list = [], []
    for t in range(CTX, len(df)):
        X_list.append(arr[t-CTX:t, :])
        y_list.append(y_all[t])

    X = np.stack(X_list).astype(np.float32)      # (N, CTX, F)
    y = np.array(y_list).astype(np.float32)      # (N,)
    return X, y

def split_tail(X, y, val_tail_frac=0.10):
    n = len(X)
    val_tail = int(n * val_tail_frac)
    Xtr, Xval = X[:-val_tail], X[-val_tail:]
    ytr, yval = y[:-val_tail], y[-val_tail:]
    return Xtr, ytr, Xval, yval

def scale_X_train_val(Xtr, Xval):
    """
    Normaliza com StandardScaler aplicado por amostra (flatten), mas com o scaler
    treinado no treino inteiro. Depois reshape.
    """
    scaler = StandardScaler()
    Xtr2 = scaler.fit_transform(Xtr.reshape(len(Xtr), -1)).reshape(Xtr.shape)
    Xval2 = scaler.transform(Xval.reshape(len(Xval), -1)).reshape(Xval.shape)
    return Xtr2.astype(np.float32), Xval2.astype(np.float32), scaler

def rmse_mae(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    return rmse, mae

# =========================
# MODELO (quant-friendly)
# =========================
def build_tinycnn(in_features: int, channels: int, l2_weight: float):
    """
    Conv1D -> ReLU -> Conv1D -> ReLU -> GlobalAveragePooling1D -> Dense(1)
    """
    reg = tf.keras.regularizers.l2(l2_weight) if l2_weight > 0 else None
    inp = tf.keras.Input(shape=(CTX, in_features), name="x")

    x = tf.keras.layers.Conv1D(
        filters=channels,
        kernel_size=KERNEL_SIZE,
        padding="same",
        use_bias=True,
        kernel_regularizer=reg,
        name="conv1",
    )(inp)
    x = tf.keras.layers.ReLU(name="relu1")(x)

    x = tf.keras.layers.Conv1D(
        filters=channels,
        kernel_size=KERNEL_SIZE,
        padding="same",
        use_bias=True,
        kernel_regularizer=reg,
        name="conv2",
    )(x)
    x = tf.keras.layers.ReLU(name="relu2")(x)

    x = tf.keras.layers.GlobalAveragePooling1D(name="gap")(x)
    out = tf.keras.layers.Dense(1, name="y")(x)

    model = tf.keras.Model(inp, out, name="TinyCNN1D")
    return model

def compile_model(model, lr: float, weight_decay: float):
    try:
        opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    except Exception:
        opt = tf.keras.optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=opt,
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model

def train_and_eval_once(
    Xtr, ytr, Xval, yval,
    channels: int, batch_size: int,
    lr: float, weight_decay: float, l2_weight: float,
    seed: int
):
    set_seeds(seed)

    in_features = Xtr.shape[-1]
    model = build_tinycnn(in_features=in_features, channels=channels, l2_weight=l2_weight)

    n_params = int(model.count_params())
    model = compile_model(model, lr=lr, weight_decay=weight_decay)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=0,
        ),
    ]

    model.fit(
        Xtr, ytr,
        validation_data=(Xval, yval),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        shuffle=False, 
        verbose=0,
        callbacks=callbacks,
    )

    preds = model.predict(Xval, batch_size=batch_size, verbose=0).reshape(-1)
    rmse, mae = rmse_mae(yval, preds)
    return model, rmse, mae, n_params

_df = load_dataframe()
_X, _y = build_sequences(_df)
_Xtr, _ytr, _Xval, _yval = split_tail(_X, _y, VAL_TAIL_FRAC)
_Xtr_s, _Xval_s, _scaler = scale_X_train_val(_Xtr, _Xval)

scaler_info = {
    "mean": _scaler.mean_.tolist(),
    "scale": _scaler.scale_.tolist(),
    "var": _scaler.var_.tolist(),
    "n_features_in": int(getattr(_scaler, "n_features_in_", _Xtr_s.reshape(len(_Xtr_s), -1).shape[1])),
    "ctx": CTX,
}
with open(os.path.join(OUT_DIR, "scaler_standard.json"), "w") as f:
    json.dump(scaler_info, f, indent=2)

# =========================
# HPO (Optuna)
# =========================
def objective(trial: optuna.Trial):
    seed = SEED + trial.number

    channels = trial.suggest_categorical("channels", [8, 16, 32, 64])
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-4, log=True)

    use_l2 = trial.suggest_categorical("use_l2", [0, 1])
    if use_l2 == 0:
        l2_weight = 0.0
    else:
        l2_weight = trial.suggest_float("l2_weight", 1e-8, 1e-4, log=True)

    model, rmse, mae, n_params = train_and_eval_once(
        _Xtr_s, _ytr, _Xval_s, _yval,
        channels=channels, batch_size=batch_size,
        lr=lr, weight_decay=weight_decay, l2_weight=l2_weight,
        seed=seed,
    )

    trial.set_user_attr("rmse", rmse)
    trial.set_user_attr("mae", mae)
    trial.set_user_attr("n_params", n_params)

    if n_params > MAX_PARAMS_BUDGET:
        raise optuna.TrialPruned(f"Excedeu budget params: {n_params} > {MAX_PARAMS_BUDGET}")

    return rmse

def run_hpo():
    study = optuna.create_study(
        direction="minimize",
        pruner=MedianPruner(n_startup_trials=10),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    with open(os.path.join(OUT_DIR, "optuna_best_params.json"), "w") as f:
        json.dump(
            {"best_rmse": float(study.best_value), "best_params": study.best_params},
            f,
            indent=2,
        )

    rows = []
    for t in study.trials:
        if t.state.name not in ("COMPLETE", "PRUNED"):
            continue
        row = {
            "trial_number": t.number,
            "state": t.state.name,
            "value_rmse_objective": t.value if t.value is not None else np.nan,
        }
        row.update(t.params)
        row["rmse"] = t.user_attrs.get("rmse", np.nan)
        row["mae"] = t.user_attrs.get("mae", np.nan)
        row["n_params"] = t.user_attrs.get("n_params", np.nan)
        rows.append(row)

    df_trials = pd.DataFrame(rows)

    state_rank = {"COMPLETE": 0, "PRUNED": 1}
    df_trials["state_rank"] = df_trials["state"].map(state_rank).fillna(9).astype(int)
    df_trials = df_trials.sort_values(["state_rank", "rmse"], ascending=[True, True]).drop(columns=["state_rank"])

    trials_csv = os.path.join(OUT_DIR, "hpo_trials_proxy.csv")
    df_trials.to_csv(trials_csv, index=False)
    print(f"[OK] Trials (proxy) salvos em: {trials_csv}")

    return study, df_trials

def convert_to_tflite_float(model: tf.keras.Model, out_path: str) -> int:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    with open(out_path, "wb") as f:
        f.write(tflite_model)
    return os.path.getsize(out_path)

def convert_to_tflite_int8_ptq(model: tf.keras.Model, out_path: str, rep_data: np.ndarray) -> int:
    """
    PTQ int8 (simples e rápido).
    Depois, dá pra trocar por QAT (melhor em MCU).
    """
    def representative_dataset():
        for i in range(len(rep_data)):
            x = rep_data[i:i+1]
            yield [x]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()
    with open(out_path, "wb") as f:
        f.write(tflite_model)
    return os.path.getsize(out_path)

def postprocess_topk_to_tflite(df_trials: pd.DataFrame, top_k: int = TOP_K):
  
    df_ok = df_trials[(df_trials["state"] == "COMPLETE") & (df_trials["n_params"] <= MAX_PARAMS_BUDGET)].copy()
    df_ok = df_ok.sort_values("rmse", ascending=True).head(top_k)

    if len(df_ok) == 0:
        print("[ERRO] Nenhum trial completo dentro do budget. Aumente MAX_PARAMS_BUDGET ou ajuste search space.")
        return None

    rep_n = min(200, len(_Xtr_s))
    rep_data = _Xtr_s[:rep_n].astype(np.float32)

    report_rows = []

    for _, row in df_ok.iterrows():
        trial_num = int(row["trial_number"])
        channels = int(row["channels"])
        batch_size = int(row["batch_size"])
        lr = float(row["lr"])
        weight_decay = float(row["weight_decay"])

        use_l2 = int(row.get("use_l2", 0))
        if use_l2 == 0:
            l2_weight = 0.0
        else:
            l2_weight = float(row["l2_weight"])

        model, rmse, mae, n_params = train_and_eval_once(
            _Xtr_s, _ytr, _Xval_s, _yval,
            channels=channels, batch_size=batch_size,
            lr=lr, weight_decay=weight_decay, l2_weight=l2_weight,
            seed=SEED + trial_num,
        )

        float_path = os.path.join(OUT_DIR, f"trial{trial_num:03d}_float32.tflite")
        float_bytes = convert_to_tflite_float(model, float_path)

        report_rows.append({
            "trial_number": trial_num,
            "variant": "baseline_float32",
            "RMSE": rmse,
            "MAE": mae,
            "Tamanho_Modelo_bytes": float_bytes,
            "N_Parametros": n_params,
            "channels": channels,
            "batch_size": batch_size,
            "lr": lr,
            "weight_decay": weight_decay,
            "use_l2": use_l2,
            "l2_weight": l2_weight,
            "tflite_path": float_path,
        })

        int8_path = os.path.join(OUT_DIR, f"trial{trial_num:03d}_int8_ptq.tflite")
        int8_bytes = convert_to_tflite_int8_ptq(model, int8_path, rep_data=rep_data)

        report_rows.append({
            "trial_number": trial_num,
            "variant": "quant_int8_ptq",
            "RMSE": rmse, 
            "MAE": mae,
            "Tamanho_Modelo_bytes": int8_bytes,
            "N_Parametros": n_params,
            "channels": channels,
            "batch_size": batch_size,
            "lr": lr,
            "weight_decay": weight_decay,
            "use_l2": use_l2,
            "l2_weight": l2_weight,
            "tflite_path": int8_path,
        })

    df_report = pd.DataFrame(report_rows)
    out_csv = os.path.join(OUT_DIR, "topk_tflite_report.csv")
    df_report.to_csv(out_csv, index=False)
    print(f"[OK] Report Top-K (TFLite) salvo em: {out_csv}")

    df_int8 = df_report[df_report["variant"] == "quant_int8_ptq"].copy()
    df_int8 = df_int8.sort_values(["Tamanho_Modelo_bytes", "RMSE"], ascending=[True, True]).reset_index(drop=True)

    best = df_int8.iloc[0].to_dict()
    with open(os.path.join(OUT_DIR, "best_candidate_int8.json"), "w") as f:
        json.dump(best, f, indent=2)

    print("[OK] Melhor candidato (prioriza tamanho -> RMSE) salvo em best_candidate_int8.json")
    return df_report

# =========================
# MAIN
# =========================
if __name__ == "__main__":
    t0 = time.time()
    study, df_trials = run_hpo()
    _ = postprocess_topk_to_tflite(df_trials, top_k=TOP_K)
    print(f"Tempo total: {time.time() - t0:.1f}s")


## CNN BASELINE

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd

import tensorflow as tf
from sklearn.preprocessing import StandardScaler

BASE_DIR = "tinyCNN"
os.makedirs(BASE_DIR, exist_ok=True)

CSV_PATH = "bd_EstacaoVargemFria_e_Pesca.csv"
STATION_NAME = "Estação Pesca - UFRPE"
TIME_COL = "Data estação"

# =========================
# Config do modelo/dados
# =========================
CTX = 7
KERNEL_SIZE = 5
VAL_TAIL_FRAC = 0.10
SEED = 42

# =========================
# Hiperparâmetros (do trial 68)
# tinyCNN baseline (float32)
# =========================
HP = {
    "channels": 16,
    "batch_size": 64,
    "lr": 0.0012533827963110565,
    "weight_decay": 9.791657848508889e-06,
    "use_l2": 0,
    "l2_weight": 0.0,
    "max_epochs": 100,
    "patience": 15,
}

# =========================
# Utils
# =========================
def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def load_dataframe():
    df = pd.read_csv(CSV_PATH, on_bad_lines="skip")
    df = df[df["Nome"].astype(str).str.strip() == STATION_NAME].copy()

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])

    cols = ["Temperatura", "Umidade", "Velocidade Vento", "Rajada Vento", "Precipitação dia"]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.sort_values(TIME_COL)
    df = df.infer_objects(copy=False).interpolate().dropna().reset_index(drop=True)
    return df

def build_sequences(df):
    cols_X = ["Temperatura", "Umidade", "Velocidade Vento", "Rajada Vento", "Precipitação dia"]
    arr = df[cols_X].values.astype(np.float32)
    y_all = df["Precipitação dia"].values.astype(np.float32)

    X_list, y_list = [], []
    for t in range(CTX, len(df)):
        X_list.append(arr[t-CTX:t, :])
        y_list.append(y_all[t])

    X = np.stack(X_list).astype(np.float32)   # (N, CTX, F)
    y = np.array(y_list).astype(np.float32)   # (N,)
    return X, y

def split_tail(X, y, val_tail_frac=0.10):
    n = len(X)
    val_tail = int(n * val_tail_frac)
    Xtr, Xval = X[:-val_tail], X[-val_tail:]
    ytr, yval = y[:-val_tail], y[-val_tail:]
    return Xtr, ytr, Xval, yval

def scale_X_train_val(Xtr, Xval):
    scaler = StandardScaler()
    Xtr2 = scaler.fit_transform(Xtr.reshape(len(Xtr), -1)).reshape(Xtr.shape)
    Xval2 = scaler.transform(Xval.reshape(len(Xval), -1)).reshape(Xval.shape)
    return Xtr2.astype(np.float32), Xval2.astype(np.float32), scaler

def rmse_mae(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    return rmse, mae

# =========================
# Modelo 
# =========================
def build_tinycnn(in_features: int, channels: int, l2_weight: float):
    reg = tf.keras.regularizers.l2(l2_weight) if l2_weight > 0 else None
    inp = tf.keras.Input(shape=(CTX, in_features), name="x")

    x = tf.keras.layers.Conv1D(
        filters=channels,
        kernel_size=KERNEL_SIZE,
        padding="same",
        use_bias=True,
        kernel_regularizer=reg,
        name="conv1",
    )(inp)
    x = tf.keras.layers.ReLU(name="relu1")(x)

    x = tf.keras.layers.Conv1D(
        filters=channels,
        kernel_size=KERNEL_SIZE,
        padding="same",
        use_bias=True,
        kernel_regularizer=reg,
        name="conv2",
    )(x)
    x = tf.keras.layers.ReLU(name="relu2")(x)

    x = tf.keras.layers.GlobalAveragePooling1D(name="gap")(x)
    out = tf.keras.layers.Dense(1, name="y")(x)

    return tf.keras.Model(inp, out, name="TinyCNN1D_Puro")

def compile_model(model, lr: float, weight_decay: float):

    try:
        opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    except Exception:
        opt = tf.keras.optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=opt,
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model

def export_tflite_float32(model: tf.keras.Model, out_path: str) -> int:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    with open(out_path, "wb") as f:
        f.write(tflite_model)
    return os.path.getsize(out_path)

def bytes_to_c_array(data: bytes, var_name: str = "g_model") -> str:
    
    hex_bytes = [f"0x{b:02x}" for b in data]
    lines = []
    lines.append("#pragma once")
    lines.append("#include <cstdint>")
    lines.append("")
    lines.append(f"// Model data: {len(data)} bytes")
    lines.append(f"alignas(16) const unsigned char {var_name}[] = {{")
  
    for i in range(0, len(hex_bytes), 12):
        chunk = ", ".join(hex_bytes[i:i+12])
        lines.append(f"  {chunk},")
    lines.append("};")
    lines.append(f"const unsigned int {var_name}_len = {len(data)};")
    lines.append("")
    return "\n".join(lines)

def export_model_h_from_tflite(tflite_path: str, out_h_path: str, var_name: str = "g_model"):
    with open(tflite_path, "rb") as f:
        data = f.read()
    header = bytes_to_c_array(data, var_name=var_name)
    with open(out_h_path, "w", encoding="utf-8") as f:
        f.write(header)

def main():
    set_seeds(SEED)

    df = load_dataframe()
    X, y = build_sequences(df)
    Xtr, ytr, Xval, yval = split_tail(X, y, VAL_TAIL_FRAC)
    Xtr_s, Xval_s, scaler = scale_X_train_val(Xtr, Xval)

    scaler_info = {
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
        "var": scaler.var_.tolist(),
        "ctx": CTX,
        "n_features_flat": int(Xtr_s.reshape(len(Xtr_s), -1).shape[1]),
    }
    scaler_path = os.path.join(BASE_DIR, "scaler_standard.json")
    with open(scaler_path, "w") as f:
        json.dump(scaler_info, f, indent=2)

    # 3) Modelo
    in_features = Xtr_s.shape[-1]
    model = build_tinycnn(in_features=in_features, channels=HP["channels"], l2_weight=HP["l2_weight"])
    model = compile_model(model, lr=HP["lr"], weight_decay=HP["weight_decay"])

    # 4) Treino + early stopping
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=int(HP["patience"]),
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(os.path.join(BASE_DIR, "train_log.csv"), append=False),
    ]

    history = model.fit(
        Xtr_s, ytr,
        validation_data=(Xval_s, yval),
        epochs=int(HP["max_epochs"]),
        batch_size=int(HP["batch_size"]),
        shuffle=False,
        verbose=1,
        callbacks=callbacks,
    )

    preds = model.predict(Xval_s, batch_size=int(HP["batch_size"]), verbose=0).reshape(-1)
    rmse, mae = rmse_mae(yval, preds)

    # 6) Export TFLite float32
    tflite_path = os.path.join(BASE_DIR, "tinycnn_puro_float32.tflite")
    tflite_bytes = export_tflite_float32(model, tflite_path)

    # 7) Export model.h para Arduino
    model_h_path = os.path.join(BASE_DIR, "model.h")
    export_model_h_from_tflite(tflite_path, model_h_path, var_name="g_tinycnn_puro")

    # 8) Metadados e métricas
    metrics = {
        "variant": "tinycnn_puro_float32",
        "RMSE_val": rmse,
        "MAE_val": mae,
        "tflite_path": tflite_path,
        "tflite_bytes": int(tflite_bytes),
        "model_h_path": model_h_path,
        "n_params": int(model.count_params()),
        "hyperparams": HP,
        "input_shape": [None, CTX, int(in_features)],
        
    }
    with open(os.path.join(BASE_DIR, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    print("\n[OK] Treino finalizado.")
    print(f"[OK] RMSE_val = {rmse:.6f} | MAE_val = {mae:.6f}")
    print(f"[OK] TFLite (float32): {tflite_path} ({tflite_bytes} bytes)")
    print(f"[OK] Header (Arduino): {model_h_path}")
    print(f"[OK] Scaler: {scaler_path}")

if __name__ == "__main__":
    main()

## QUANTIZAÇÃO PTQ

In [ ]:

import os
import json
import random
import numpy as np
import pandas as pd

import tensorflow as tf
from sklearn.preprocessing import StandardScaler


BASE_DIR = "tinyCNN_quant"
os.makedirs(BASE_DIR, exist_ok=True)

# Dataset (o mesmo que você usa)
CSV_PATH = "bd_EstacaoVargemFria_e_Pesca.csv"
STATION_NAME = "Estação Pesca - UFRPE"
TIME_COL = "Data estação"

# =========================
# Config do modelo/dados
# =========================
CTX = 7
KERNEL_SIZE = 5
VAL_TAIL_FRAC = 0.10
SEED = 42

# Quantização PTQ
REP_DATASET_SAMPLES = 300   
REP_DATASET_BATCH = 1       

HP = {
    "channels": 16,
    "batch_size": 64,
    "lr": 0.0012533827963110565,
    "weight_decay": 9.791657848508889e-06,
    "use_l2": 0,
    "l2_weight": 0.0,
    "max_epochs": 100,
    "patience": 15,
}


def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def load_dataframe():
    df = pd.read_csv(CSV_PATH, on_bad_lines="skip")
    df = df[df["Nome"].astype(str).str.strip() == STATION_NAME].copy()

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])

    cols = ["Temperatura", "Umidade", "Velocidade Vento", "Rajada Vento", "Precipitação dia"]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.sort_values(TIME_COL)
    df = df.infer_objects(copy=False).interpolate().dropna().reset_index(drop=True)
    return df

def build_sequences(df):
    cols_X = ["Temperatura", "Umidade", "Velocidade Vento", "Rajada Vento", "Precipitação dia"]
    arr = df[cols_X].values.astype(np.float32)
    y_all = df["Precipitação dia"].values.astype(np.float32)

    X_list, y_list = [], []
    for t in range(CTX, len(df)):
        X_list.append(arr[t-CTX:t, :])
        y_list.append(y_all[t])

    X = np.stack(X_list).astype(np.float32)   # (N, CTX, F)
    y = np.array(y_list).astype(np.float32)   # (N,)
    return X, y

def split_tail(X, y, val_tail_frac=0.10):
    n = len(X)
    val_tail = int(n * val_tail_frac)
    Xtr, Xval = X[:-val_tail], X[-val_tail:]
    ytr, yval = y[:-val_tail], y[-val_tail:]
    return Xtr, ytr, Xval, yval

def scale_X_train_val(Xtr, Xval):
    scaler = StandardScaler()
    Xtr2 = scaler.fit_transform(Xtr.reshape(len(Xtr), -1)).reshape(Xtr.shape)
    Xval2 = scaler.transform(Xval.reshape(len(Xval), -1)).reshape(Xval.shape)
    return Xtr2.astype(np.float32), Xval2.astype(np.float32), scaler

def rmse_mae(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    return rmse, mae


# =========================
# Modelo 
# =========================
def build_tinycnn(in_features: int, channels: int, l2_weight: float):
    reg = tf.keras.regularizers.l2(l2_weight) if l2_weight > 0 else None
    inp = tf.keras.Input(shape=(CTX, in_features), name="x")

    x = tf.keras.layers.Conv1D(
        filters=channels,
        kernel_size=KERNEL_SIZE,
        padding="same",
        use_bias=True,
        kernel_regularizer=reg,
        name="conv1",
    )(inp)
    x = tf.keras.layers.ReLU(name="relu1")(x)

    x = tf.keras.layers.Conv1D(
        filters=channels,
        kernel_size=KERNEL_SIZE,
        padding="same",
        use_bias=True,
        kernel_regularizer=reg,
        name="conv2",
    )(x)
    x = tf.keras.layers.ReLU(name="relu2")(x)

    x = tf.keras.layers.GlobalAveragePooling1D(name="gap")(x)
    out = tf.keras.layers.Dense(1, name="y")(x)

    return tf.keras.Model(inp, out, name="TinyCNN1D_Puro")

def compile_model(model, lr: float, weight_decay: float):
    try:
        opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    except Exception:
        opt = tf.keras.optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=opt,
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


# =========================
# Export: TFLite float32
# =========================
def export_tflite_float32(model: tf.keras.Model, out_path: str) -> int:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [] 
    tflite_model = converter.convert()
    with open(out_path, "wb") as f:
        f.write(tflite_model)
    return os.path.getsize(out_path)


# =========================
# Export: TFLite PTQ int8 (Arduino)
# =========================
def make_representative_dataset(X_samples: np.ndarray, batch_size: int = 1):
    """
    Gera um generator para calibração do PTQ.
    """
    X_samples = X_samples.astype(np.float32)

    def gen():
        for i in range(len(X_samples)):
            x = X_samples[i:i+1]  # (1, CTX, F)
            yield [x]
    return gen

def export_tflite_ptq_int8(model: tf.keras.Model, Xtr_s: np.ndarray, out_path: str,
                          n_samples: int = 300) -> int:
    """
    PTQ int8 full-integer:
    - calibrado com representative_dataset
    - input/output int8
    """
    n = len(Xtr_s)
    if n == 0:
        raise ValueError("Xtr_s vazio - não dá pra calibrar PTQ.")
    n_use = min(n_samples, n)
    idx = np.linspace(0, n - 1, num=n_use, dtype=int)  
    X_rep = Xtr_s[idx].astype(np.float32)

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = make_representative_dataset(X_rep, batch_size=REP_DATASET_BATCH)

    # força kernels int8
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

    # input/output int8 
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()

    with open(out_path, "wb") as f:
        f.write(tflite_model)

    return os.path.getsize(out_path)


def extract_io_quant_params(tflite_path: str) -> dict:
    """
    Extrai scale/zero_point do tensor de entrada e saída do modelo TFLite.
    """
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    in_details = interpreter.get_input_details()[0]
    out_details = interpreter.get_output_details()[0]

    in_scale, in_zero = in_details.get("quantization", (None, None))
    out_scale, out_zero = out_details.get("quantization", (None, None))

    info = {
        "input": {
            "name": in_details.get("name"),
            "shape": [int(x) for x in in_details.get("shape")],
            "dtype": str(in_details.get("dtype")),
            "quantization": {
                "scale": float(in_scale) if in_scale is not None else None,
                "zero_point": int(in_zero) if in_zero is not None else None,
            },
        },
        "output": {
            "name": out_details.get("name"),
            "shape": [int(x) for x in out_details.get("shape")],
            "dtype": str(out_details.get("dtype")),
            "quantization": {
                "scale": float(out_scale) if out_scale is not None else None,
                "zero_point": int(out_zero) if out_zero is not None else None,
            },
        },
         }
    return info


# =========================
# Export: model.h (C array)
# =========================
def bytes_to_c_array(data: bytes, var_name: str = "g_model") -> str:
    hex_bytes = [f"0x{b:02x}" for b in data]
    lines = []
    lines.append("#pragma once")
    lines.append("#include <cstdint>")
    lines.append("")
    lines.append(f"// Model data: {len(data)} bytes")
    lines.append(f"alignas(16) const unsigned char {var_name}[] = {{")
    for i in range(0, len(hex_bytes), 12):
        chunk = ", ".join(hex_bytes[i:i+12])
        lines.append(f"  {chunk},")
    lines.append("};")
    lines.append(f"const unsigned int {var_name}_len = {len(data)};")
    lines.append("")
    return "\n".join(lines)

def export_model_h_from_tflite(tflite_path: str, out_h_path: str, var_name: str = "g_model"):
    with open(tflite_path, "rb") as f:
        data = f.read()
    header = bytes_to_c_array(data, var_name=var_name)
    with open(out_h_path, "w", encoding="utf-8") as f:
        f.write(header)


# =========================
# MAIN
# =========================
def main():
    set_seeds(SEED)

    df = load_dataframe()
    X, y = build_sequences(df)
    Xtr, ytr, Xval, yval = split_tail(X, y, VAL_TAIL_FRAC)
    Xtr_s, Xval_s, scaler = scale_X_train_val(Xtr, Xval)

    scaler_info = {
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
        "var": scaler.var_.tolist(),
        "ctx": CTX,
        "n_features_flat": int(Xtr_s.reshape(len(Xtr_s), -1).shape[1]),
    }
    scaler_path = os.path.join(BASE_DIR, "scaler_standard.json")
    with open(scaler_path, "w") as f:
        json.dump(scaler_info, f, indent=2)

    # 3) Modelo
    in_features = Xtr_s.shape[-1]
    model = build_tinycnn(in_features=in_features, channels=HP["channels"], l2_weight=HP["l2_weight"])
    model = compile_model(model, lr=HP["lr"], weight_decay=HP["weight_decay"])

    # 4) Treino + early stopping
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=int(HP["patience"]),
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(os.path.join(BASE_DIR, "train_log.csv"), append=False),
    ]

    model.fit(
        Xtr_s, ytr,
        validation_data=(Xval_s, yval),
        epochs=int(HP["max_epochs"]),
        batch_size=int(HP["batch_size"]),
        shuffle=False,
        verbose=1,
        callbacks=callbacks,
    )

    preds = model.predict(Xval_s, batch_size=int(HP["batch_size"]), verbose=0).reshape(-1)
    rmse, mae = rmse_mae(yval, preds)

    # -------------------------
    # 6) Export baseline float32
    # -------------------------
    tflite_float_path = os.path.join(BASE_DIR, "tinycnn_puro_float32.tflite")
    tflite_float_bytes = export_tflite_float32(model, tflite_float_path)

    model_h_float = os.path.join(BASE_DIR, "model_puro.h")
    export_model_h_from_tflite(tflite_float_path, model_h_float, var_name="g_tinycnn_puro")

    # -------------------------
    # 7) Export PTQ int8
    # -------------------------
    tflite_int8_path = os.path.join(BASE_DIR, "tinycnn_ptq_int8.tflite")
    tflite_int8_bytes = export_tflite_ptq_int8(
        model,
        Xtr_s=Xtr_s,
        out_path=tflite_int8_path,
        n_samples=REP_DATASET_SAMPLES,
    )

    model_h_int8 = os.path.join(BASE_DIR, "model_ptq_int8.h")
    export_model_h_from_tflite(tflite_int8_path, model_h_int8, var_name="g_tinycnn_ptq_int8")

    quant_params = extract_io_quant_params(tflite_int8_path)
    quant_params_path = os.path.join(BASE_DIR, "quant_params_ptq_int8.json")
    with open(quant_params_path, "w") as f:
        json.dump(quant_params, f, indent=2)

    metrics = {
        "variant_baseline": "tinycnn_puro_float32",
        "variant_ptq": "tinycnn_ptq_int8",
        "RMSE_val_float32_keras": rmse,
        "MAE_val_float32_keras": mae,
        "tflite_float32": {
            "path": tflite_float_path,
            "bytes": int(tflite_float_bytes),
            "model_h": model_h_float,
            "c_var": "g_tinycnn_puro",
        },
        "tflite_ptq_int8": {
            "path": tflite_int8_path,
            "bytes": int(tflite_int8_bytes),
            "model_h": model_h_int8,
            "c_var": "g_tinycnn_ptq_int8",
            "quant_params_path": quant_params_path,
        },
        "n_params": int(model.count_params()),
        "hyperparams": HP,
        "input_shape": [None, CTX, int(in_features)],
        
    }
    metrics_path = os.path.join(BASE_DIR, "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    print("\n[OK] Treino finalizado.")
    print(f"[OK] RMSE_val (keras/float32) = {rmse:.6f} | MAE_val = {mae:.6f}")
    print(f"[OK] TFLite baseline float32: {tflite_float_path} ({tflite_float_bytes} bytes)")
    print(f"[OK] Header baseline (Arduino): {model_h_float}")
    print(f"[OK] TFLite PTQ int8: {tflite_int8_path} ({tflite_int8_bytes} bytes)")
    print(f"[OK] Header PTQ int8 (Arduino): {model_h_int8}")
    print(f"[OK] Quant params PTQ: {quant_params_path}")
    print(f"[OK] Scaler: {scaler_path}")
    print(f"[OK] Metrics: {metrics_path}")


if __name__ == "__main__":
    main()

## PODA

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd

import tensorflow as tf
from sklearn.preprocessing import StandardScaler

import tensorflow_model_optimization as tfmot


# =========================
# PATHS
# =========================
BASE_DIR = "tinyCNN_poda"
os.makedirs(BASE_DIR, exist_ok=True)

CSV_PATH = "bd_EstacaoVargemFria_e_Pesca.csv"
STATION_NAME = "Estação Pesca - UFRPE"
TIME_COL = "Data estação"


# =========================
# CONFIG DADOS/MODELO
# =========================
CTX = 7
KERNEL_SIZE = 5
VAL_TAIL_FRAC = 0.10
SEED = 42

REP_DATASET_SAMPLES = 300
REP_DATASET_BATCH = 1


# =========================
# HIPERPARÂMETROS BASE
# =========================
HP = {
    "channels": 16,
    "batch_size": 64,
    "lr": 0.0012533827963110565,
    "weight_decay": 9.791657848508889e-06,
    "use_l2": 0,
    "l2_weight": 0.0,
    "max_epochs": 100,
    "patience": 15,
}


# =========================
# HIPERPARÂMETROS DA PODA
# =========================
PRUNE_CFG = {
    "initial_sparsity": 0.10,
    "final_sparsity": 0.50,
    "begin_step": 0,
    "frequency": 100,
}


# =========================
# UTILS
# =========================
def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def load_dataframe():
    df = pd.read_csv(CSV_PATH, on_bad_lines="skip")
    df = df[df["Nome"].astype(str).str.strip() == STATION_NAME].copy()

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])

    cols = ["Temperatura", "Umidade", "Velocidade Vento", "Rajada Vento", "Precipitação dia"]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.sort_values(TIME_COL)
    df = df.infer_objects(copy=False).interpolate().dropna().reset_index(drop=True)
    return df

def build_sequences(df):
    cols_X = ["Temperatura", "Umidade", "Velocidade Vento", "Rajada Vento", "Precipitação dia"]
    arr = df[cols_X].values.astype(np.float32)
    y_all = df["Precipitação dia"].values.astype(np.float32)

    X_list, y_list = [], []
    for t in range(CTX, len(df)):
        X_list.append(arr[t-CTX:t, :])  # (CTX, F)
        y_list.append(y_all[t])

    X = np.stack(X_list).astype(np.float32)   # (N, CTX, F)
    y = np.array(y_list).astype(np.float32)   # (N,)
    return X, y

def split_tail(X, y, val_tail_frac=0.10):
    n = len(X)
    val_tail = int(n * val_tail_frac)
    Xtr, Xval = X[:-val_tail], X[-val_tail:]
    ytr, yval = y[:-val_tail], y[-val_tail:]
    return Xtr, ytr, Xval, yval

def scale_X_train_val(Xtr, Xval):
    scaler = StandardScaler()
    Xtr2 = scaler.fit_transform(Xtr.reshape(len(Xtr), -1)).reshape(Xtr.shape)
    Xval2 = scaler.transform(Xval.reshape(len(Xval), -1)).reshape(Xval.shape)
    return Xtr2.astype(np.float32), Xval2.astype(np.float32), scaler

def rmse_mae(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    return rmse, mae


# =========================
# MODELO BASE
# =========================
def build_tinycnn(in_features: int, channels: int, l2_weight: float):
    reg = tf.keras.regularizers.l2(l2_weight) if l2_weight > 0 else None

    inp = tf.keras.Input(shape=(CTX, in_features), name="x")

    x = tf.keras.layers.Conv1D(
        filters=channels,
        kernel_size=KERNEL_SIZE,
        padding="same",
        use_bias=True,
        kernel_regularizer=reg,
        name="conv1",
    )(inp)
    x = tf.keras.layers.ReLU(name="relu1")(x)

    x = tf.keras.layers.Conv1D(
        filters=channels,
        kernel_size=KERNEL_SIZE,
        padding="same",
        use_bias=True,
        kernel_regularizer=reg,
        name="conv2",
    )(x)
    x = tf.keras.layers.ReLU(name="relu2")(x)

    x = tf.keras.layers.GlobalAveragePooling1D(name="gap")(x)
    out = tf.keras.layers.Dense(1, name="y")(x)

    return tf.keras.Model(inp, out, name="TinyCNN1D")

def compile_model(model, lr: float, weight_decay: float):
    try:
        opt = tf.keras.optimizers.AdamW(
            learning_rate=lr,
            weight_decay=weight_decay
        )
    except Exception:
        opt = tf.keras.optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=opt,
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


# =========================
# PODA
# =========================
def build_pruned_model(in_features: int, channels: int, l2_weight: float,
                       x_train_len: int, batch_size: int, epochs: int):
    """
    Cria o modelo com wrapper de pruning.
    """
    base_model = build_tinycnn(
        in_features=in_features,
        channels=channels,
        l2_weight=l2_weight
    )

    end_step = int(np.ceil(x_train_len / batch_size)) * epochs

    pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=PRUNE_CFG["initial_sparsity"],
        final_sparsity=PRUNE_CFG["final_sparsity"],
        begin_step=PRUNE_CFG["begin_step"],
        end_step=end_step,
        frequency=PRUNE_CFG["frequency"],
    )

    pruned_model = tfmot.sparsity.keras.prune_low_magnitude(
        base_model,
        pruning_schedule=pruning_schedule
    )

    pruned_model = compile_model(
        pruned_model,
        lr=HP["lr"],
        weight_decay=HP["weight_decay"]
    )

    return pruned_model, end_step


def compute_model_sparsity(stripped_model: tf.keras.Model) -> dict:
    """
    Calcula sparsity real dos pesos após strip_pruning.
    """
    total_params = 0
    zero_params = 0
    layer_stats = []

    for layer in stripped_model.layers:
        weights = layer.get_weights()
        if not weights:
            continue

        layer_total = 0
        layer_zero = 0

        for w in weights:
            arr = np.asarray(w)
            layer_total += arr.size
            layer_zero += np.sum(arr == 0)

        total_params += layer_total
        zero_params += layer_zero

        layer_stats.append({
            "layer_name": layer.name,
            "total_params": int(layer_total),
            "zero_params": int(layer_zero),
            "sparsity": float(layer_zero / layer_total) if layer_total > 0 else 0.0,
        })

    global_sparsity = float(zero_params / total_params) if total_params > 0 else 0.0

    return {
        "global_total_params": int(total_params),
        "global_zero_params": int(zero_params),
        "global_sparsity": global_sparsity,
        "per_layer": layer_stats,
    }


# =========================
# EXPORT TFLITE
# =========================
def export_tflite_float32(model: tf.keras.Model, out_path: str) -> int:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = []
    tflite_model = converter.convert()

    with open(out_path, "wb") as f:
        f.write(tflite_model)

    return os.path.getsize(out_path)

def make_representative_dataset(X_samples: np.ndarray, batch_size: int = 1):
    X_samples = X_samples.astype(np.float32)

    def gen():
        for i in range(len(X_samples)):
            x = X_samples[i:i+1]
            yield [x]
    return gen

def export_tflite_ptq_int8(model: tf.keras.Model, Xtr_s: np.ndarray, out_path: str,
                           n_samples: int = 300) -> int:
    n = len(Xtr_s)
    if n == 0:
        raise ValueError("Xtr_s vazio - não dá pra calibrar PTQ.")

    n_use = min(n_samples, n)
    idx = np.linspace(0, n - 1, num=n_use, dtype=int)
    X_rep = Xtr_s[idx].astype(np.float32)

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = make_representative_dataset(
        X_rep,
        batch_size=REP_DATASET_BATCH
    )
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()

    with open(out_path, "wb") as f:
        f.write(tflite_model)

    return os.path.getsize(out_path)


# =========================
# QUANT PARAMS
# =========================
def extract_io_quant_params(tflite_path: str) -> dict:
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    in_details = interpreter.get_input_details()[0]
    out_details = interpreter.get_output_details()[0]

    in_scale, in_zero = in_details.get("quantization", (None, None))
    out_scale, out_zero = out_details.get("quantization", (None, None))

    return {
        "input": {
            "name": in_details.get("name"),
            "shape": [int(x) for x in in_details.get("shape")],
            "dtype": str(in_details.get("dtype")),
            "quantization": {
                "scale": float(in_scale) if in_scale is not None else None,
                "zero_point": int(in_zero) if in_zero is not None else None,
            },
        },
        "output": {
            "name": out_details.get("name"),
            "shape": [int(x) for x in out_details.get("shape")],
            "dtype": str(out_details.get("dtype")),
            "quantization": {
                "scale": float(out_scale) if out_scale is not None else None,
                "zero_point": int(out_zero) if out_zero is not None else None,
            },
        },
     }


# =========================
# EXPORT HEADER .H
# =========================
def bytes_to_c_array(data: bytes, var_name: str = "g_model") -> str:
    hex_bytes = [f"0x{b:02x}" for b in data]

    lines = []
    lines.append("#pragma once")
    lines.append("#include <cstdint>")
    lines.append("")
    lines.append(f"// Model data: {len(data)} bytes")
    lines.append(f"alignas(16) const unsigned char {var_name}[] = {{")

    for i in range(0, len(hex_bytes), 12):
        chunk = ", ".join(hex_bytes[i:i+12])
        lines.append(f"  {chunk},")

    lines.append("};")
    lines.append(f"const unsigned int {var_name}_len = {len(data)};")
    lines.append("")
    return "\n".join(lines)

def export_model_h_from_tflite(tflite_path: str, out_h_path: str, var_name: str = "g_model"):
    with open(tflite_path, "rb") as f:
        data = f.read()

    header = bytes_to_c_array(data, var_name=var_name)

    with open(out_h_path, "w", encoding="utf-8") as f:
        f.write(header)


# =========================
# TREINO MODELO PURO
# =========================
def train_pure_model(Xtr_s, ytr, Xval_s, yval, in_features):
    model = build_tinycnn(
        in_features=in_features,
        channels=HP["channels"],
        l2_weight=HP["l2_weight"]
    )
    model = compile_model(model, lr=HP["lr"], weight_decay=HP["weight_decay"])

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=int(HP["patience"]),
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(
            os.path.join(BASE_DIR, "train_log_puro.csv"),
            append=False
        ),
    ]

    model.fit(
        Xtr_s, ytr,
        validation_data=(Xval_s, yval),
        epochs=int(HP["max_epochs"]),
        batch_size=int(HP["batch_size"]),
        shuffle=False,
        verbose=1,
        callbacks=callbacks,
    )

    preds = model.predict(Xval_s, batch_size=int(HP["batch_size"]), verbose=0).reshape(-1)
    rmse, mae = rmse_mae(yval, preds)

    return model, rmse, mae


# =========================
# TREINO MODELO PODADO
# =========================
def train_pruned_model(Xtr_s, ytr, Xval_s, yval, in_features):
    pruned_model, end_step = build_pruned_model(
        in_features=in_features,
        channels=HP["channels"],
        l2_weight=HP["l2_weight"],
        x_train_len=len(Xtr_s),
        batch_size=int(HP["batch_size"]),
        epochs=int(HP["max_epochs"]),
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=int(HP["patience"]),
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(
            os.path.join(BASE_DIR, "train_log_poda.csv"),
            append=False
        ),
        tfmot.sparsity.keras.UpdatePruningStep(),
    ]

    pruned_model.fit(
        Xtr_s, ytr,
        validation_data=(Xval_s, yval),
        epochs=int(HP["max_epochs"]),
        batch_size=int(HP["batch_size"]),
        shuffle=False,
        verbose=1,
        callbacks=callbacks,
    )

    stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

    stripped_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=HP["lr"]),
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )

    preds = stripped_model.predict(
        Xval_s,
        batch_size=int(HP["batch_size"]),
        verbose=0
    ).reshape(-1)

    rmse, mae = rmse_mae(yval, preds)
    sparsity_info = compute_model_sparsity(stripped_model)

    return stripped_model, rmse, mae, sparsity_info, end_step


# =========================
# MAIN
# =========================
def main():
    set_seeds(SEED)

    # 1) Dados
    df = load_dataframe()
    X, y = build_sequences(df)
    Xtr, ytr, Xval, yval = split_tail(X, y, VAL_TAIL_FRAC)
    Xtr_s, Xval_s, scaler = scale_X_train_val(Xtr, Xval)

    # 2) Scaler
    scaler_info = {
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
        "var": scaler.var_.tolist(),
        "ctx": CTX,
        "n_features_flat": int(Xtr_s.reshape(len(Xtr_s), -1).shape[1]),
    }

    scaler_path = os.path.join(BASE_DIR, "scaler_standard.json")
    with open(scaler_path, "w") as f:
        json.dump(scaler_info, f, indent=2)

    in_features = Xtr_s.shape[-1]

    # =========================================================
    # 3) MODELO PURO
    # =========================================================
    model_puro, rmse_puro, mae_puro = train_pure_model(
        Xtr_s, ytr, Xval_s, yval, in_features
    )

    tflite_puro_path = os.path.join(BASE_DIR, "tinycnn_puro_float32.tflite")
    tflite_puro_bytes = export_tflite_float32(model_puro, tflite_puro_path)

    model_h_puro = os.path.join(BASE_DIR, "model_puro.h")
    export_model_h_from_tflite(
        tflite_puro_path,
        model_h_puro,
        var_name="g_tinycnn_puro"
    )

    # =========================================================
    # 4) MODELO QUANTIZADO PTQ
    # =========================================================
    tflite_int8_path = os.path.join(BASE_DIR, "tinycnn_ptq_int8.tflite")
    tflite_int8_bytes = export_tflite_ptq_int8(
        model=model_puro,
        Xtr_s=Xtr_s,
        out_path=tflite_int8_path,
        n_samples=REP_DATASET_SAMPLES,
    )

    model_h_int8 = os.path.join(BASE_DIR, "model_ptq_int8.h")
    export_model_h_from_tflite(
        tflite_int8_path,
        model_h_int8,
        var_name="g_tinycnn_ptq_int8"
    )

    quant_params = extract_io_quant_params(tflite_int8_path)
    quant_params_path = os.path.join(BASE_DIR, "quant_params_ptq_int8.json")
    with open(quant_params_path, "w") as f:
        json.dump(quant_params, f, indent=2)

    # =========================================================
    # 5) MODELO PODADO
    # =========================================================
    model_podado, rmse_podado, mae_podado, sparsity_info, prune_end_step = train_pruned_model(
        Xtr_s, ytr, Xval_s, yval, in_features
    )

    tflite_podado_path = os.path.join(BASE_DIR, "tinycnn_podado_float32.tflite")
    tflite_podado_bytes = export_tflite_float32(model_podado, tflite_podado_path)

    model_h_podado = os.path.join(BASE_DIR, "model_podado.h")
    export_model_h_from_tflite(
        tflite_podado_path,
        model_h_podado,
        var_name="g_tinycnn_podado"
    )

    # =========================================================
    # 6) METRICS
    # =========================================================
    metrics = {
        "variant_baseline": "tinycnn_puro_float32",
        "variant_ptq": "tinycnn_ptq_int8",
        "variant_pruned": "tinycnn_podado_float32",

        "RMSE_val_float32_keras_puro": rmse_puro,
        "MAE_val_float32_keras_puro": mae_puro,

        "RMSE_val_float32_keras_podado": rmse_podado,
        "MAE_val_float32_keras_podado": mae_podado,

        "tflite_float32": {
            "path": tflite_puro_path,
            "bytes": int(tflite_puro_bytes),
            "model_h": model_h_puro,
            "c_var": "g_tinycnn_puro",
        },

        "tflite_ptq_int8": {
            "path": tflite_int8_path,
            "bytes": int(tflite_int8_bytes),
            "model_h": model_h_int8,
            "c_var": "g_tinycnn_ptq_int8",
            "quant_params_path": quant_params_path,
        },

        "tflite_pruned_float32": {
            "path": tflite_podado_path,
            "bytes": int(tflite_podado_bytes),
            "model_h": model_h_podado,
            "c_var": "g_tinycnn_podado",
        },

        "n_params_puro": int(model_puro.count_params()),
        "n_params_podado_strip": int(model_podado.count_params()),

        "hyperparams": HP,
        "pruning_config": {
            **PRUNE_CFG,
            "end_step": int(prune_end_step),
        },

        "pruning_sparsity": sparsity_info,

        "input_shape": [None, CTX, int(in_features)],

        "artifacts": {
            "scaler_standard_json": scaler_path,
            "train_log_puro_csv": os.path.join(BASE_DIR, "train_log_puro.csv"),
            "train_log_poda_csv": os.path.join(BASE_DIR, "train_log_poda.csv"),
        },

    }

    metrics_path = os.path.join(BASE_DIR, "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    # =========================================================
    # 7) PRINT FINAL
    # =========================================================
    print("\n[OK] Processo finalizado com sucesso.\n")

    print("=== PURO ===")
    print(f"[OK] RMSE_val = {rmse_puro:.6f} | MAE_val = {mae_puro:.6f}")
    print(f"[OK] TFLite puro: {tflite_puro_path} ({tflite_puro_bytes} bytes)")
    print(f"[OK] Header puro: {model_h_puro}")

    print("\n=== PTQ INT8 ===")
    print(f"[OK] TFLite PTQ int8: {tflite_int8_path} ({tflite_int8_bytes} bytes)")
    print(f"[OK] Header PTQ int8: {model_h_int8}")
    print(f"[OK] Quant params: {quant_params_path}")

    print("\n=== PODADO ===")
    print(f"[OK] RMSE_val = {rmse_podado:.6f} | MAE_val = {mae_podado:.6f}")
    print(f"[OK] TFLite podado: {tflite_podado_path} ({tflite_podado_bytes} bytes)")
    print(f"[OK] Header podado: {model_h_podado}")
    print(f"[OK] Sparsity global: {100.0 * sparsity_info['global_sparsity']:.2f}%")

    print("\n=== ARQUIVOS AUXILIARES ===")
    print(f"[OK] Scaler: {scaler_path}")
    print(f"[OK] Metrics: {metrics_path}")
    print(f"[OK] Log treino puro: {os.path.join(BASE_DIR, 'train_log_puro.csv')}")
    print(f"[OK] Log treino poda: {os.path.join(BASE_DIR, 'train_log_poda.csv')}")


if __name__ == "__main__":
    main()

## KD

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler


# =========================
# PATHS
# =========================
BASE_DIR = "tinyCNN_kd"
os.makedirs(BASE_DIR, exist_ok=True)

CSV_PATH = "bd_EstacaoVargemFria_e_Pesca.csv"
STATION_NAME = "Estação Pesca - UFRPE"
TIME_COL = "Data estação"

# =========================
# CONFIG GERAL
# =========================
CTX = 7
KERNEL_SIZE = 5
VAL_TAIL_FRAC = 0.10
SEED = 42

REP_DATASET_SAMPLES = 300
REP_DATASET_BATCH = 1

# =========================
# HP do teacher (igual ao baseline)
# =========================
HP_TEACHER = {
    "channels": 16,
    "batch_size": 64,
    "lr": 0.0012533827963110565,
    "weight_decay": 9.791657848508889e-06,
    "use_l2": 0,
    "l2_weight": 0.0,
    "max_epochs": 100,
    "patience": 15,
}

# =========================
# HP do student KD
# =========================
HP_STUDENT = {
    "channels": 8,
    "batch_size": 64,
    "lr": 0.0012533827963110565,
    "weight_decay": 9.791657848508889e-06,
    "use_l2": 0,
    "l2_weight": 0.0,
    "max_epochs": 100,
    "patience": 15,
}

# =========================
# HP da distilação
# =========================
KD_CONFIG = {
    "alpha": 0.7,        
    "temperature": 3.0,  
}


# =========================
# UTILS
# =========================
def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def load_dataframe():
    df = pd.read_csv(CSV_PATH, on_bad_lines="skip")
    df = df[df["Nome"].astype(str).str.strip() == STATION_NAME].copy()

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])

    cols = ["Temperatura", "Umidade", "Velocidade Vento", "Rajada Vento", "Precipitação dia"]
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.sort_values(TIME_COL)
    df = df.infer_objects(copy=False).interpolate().dropna().reset_index(drop=True)
    return df

def build_sequences(df):
    cols_X = ["Temperatura", "Umidade", "Velocidade Vento", "Rajada Vento", "Precipitação dia"]
    arr = df[cols_X].values.astype(np.float32)
    y_all = df["Precipitação dia"].values.astype(np.float32)

    X_list, y_list = [], []
    for t in range(CTX, len(df)):
        X_list.append(arr[t-CTX:t, :])  # (CTX, F)
        y_list.append(y_all[t])

    X = np.stack(X_list).astype(np.float32)
    y = np.array(y_list).astype(np.float32)
    return X, y

def split_tail(X, y, val_tail_frac=0.10):
    n = len(X)
    val_tail = int(n * val_tail_frac)
    Xtr, Xval = X[:-val_tail], X[-val_tail:]
    ytr, yval = y[:-val_tail], y[-val_tail:]
    return Xtr, ytr, Xval, yval

def scale_X_train_val(Xtr, Xval):
    scaler = StandardScaler()
    Xtr2 = scaler.fit_transform(Xtr.reshape(len(Xtr), -1)).reshape(Xtr.shape)
    Xval2 = scaler.transform(Xval.reshape(len(Xval), -1)).reshape(Xval.shape)
    return Xtr2.astype(np.float32), Xval2.astype(np.float32), scaler

def rmse_mae(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    return rmse, mae


# =========================
# CALLBACK para CSV manual
# =========================
class SimpleCSVLogger(tf.keras.callbacks.Callback):
    def __init__(self, filepath):
        super().__init__()
        self.filepath = filepath
        self.header_written = False

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        row = {"epoch": epoch + 1}
        for k, v in logs.items():
            try:
                row[k] = float(v)
            except Exception:
                row[k] = v

        df_row = pd.DataFrame([row])

        if not self.header_written:
            df_row.to_csv(self.filepath, index=False, mode="w")
            self.header_written = True
        else:
            df_row.to_csv(self.filepath, index=False, mode="a", header=False)


# =========================
# MODELO
# =========================
def build_tinycnn(in_features: int, channels: int, l2_weight: float, name="TinyCNN1D"):
    reg = tf.keras.regularizers.l2(l2_weight) if l2_weight > 0 else None
    inp = tf.keras.Input(shape=(CTX, in_features), name="x")

    x = tf.keras.layers.Conv1D(
        filters=channels,
        kernel_size=KERNEL_SIZE,
        padding="same",
        use_bias=True,
        kernel_regularizer=reg,
        name="conv1",
    )(inp)
    x = tf.keras.layers.ReLU(name="relu1")(x)

    x = tf.keras.layers.Conv1D(
        filters=channels,
        kernel_size=KERNEL_SIZE,
        padding="same",
        use_bias=True,
        kernel_regularizer=reg,
        name="conv2",
    )(x)
    x = tf.keras.layers.ReLU(name="relu2")(x)

    x = tf.keras.layers.GlobalAveragePooling1D(name="gap")(x)
    out = tf.keras.layers.Dense(1, name="y")(x)

    return tf.keras.Model(inp, out, name=name)

def compile_model(model, lr: float, weight_decay: float):
    try:
        opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    except Exception:
        opt = tf.keras.optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=opt,
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


# =========================
# DISTILLER PARA REGRESSÃO
# =========================
class RegressionDistiller(tf.keras.Model):
    def __init__(self, student, teacher, alpha=0.7, temperature=3.0):
        super().__init__()
        self.teacher = teacher
        self.student = student
        self.alpha = alpha
        self.temperature = temperature

        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.student_loss_tracker = tf.keras.metrics.Mean(name="student_loss")
        self.distill_loss_tracker = tf.keras.metrics.Mean(name="distill_loss")
        self.mae_tracker = tf.keras.metrics.MeanAbsoluteError(name="mae")

    @property
    def metrics(self):
        return [
            self.loss_tracker,
            self.student_loss_tracker,
            self.distill_loss_tracker,
            self.mae_tracker,
        ]

    def compile(self, optimizer, student_loss_fn, distillation_loss_fn):
        super().compile()
        self.optimizer = optimizer
        self.student_loss_fn = student_loss_fn
        self.distillation_loss_fn = distillation_loss_fn

    def train_step(self, data):
        x, y = data

        teacher_pred = tf.stop_gradient(self.teacher(x, training=False))

        with tf.GradientTape() as tape:
            student_pred = self.student(x, training=True)

            student_loss = self.student_loss_fn(y, student_pred)

            teacher_soft = teacher_pred / self.temperature
            student_soft = student_pred / self.temperature
            distill_loss = self.distillation_loss_fn(teacher_soft, student_soft)

            loss = self.alpha * student_loss + (1.0 - self.alpha) * distill_loss

        trainable_vars = self.student.trainable_variables
        grads = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(grads, trainable_vars))

        self.loss_tracker.update_state(loss)
        self.student_loss_tracker.update_state(student_loss)
        self.distill_loss_tracker.update_state(distill_loss)
        self.mae_tracker.update_state(y, student_pred)

        return {
            "loss": self.loss_tracker.result(),
            "student_loss": self.student_loss_tracker.result(),
            "distill_loss": self.distill_loss_tracker.result(),
            "mae": self.mae_tracker.result(),
        }

    def test_step(self, data):
        x, y = data
        student_pred = self.student(x, training=False)

        student_loss = self.student_loss_fn(y, student_pred)
        loss = student_loss

        self.loss_tracker.update_state(loss)
        self.student_loss_tracker.update_state(student_loss)
        self.distill_loss_tracker.update_state(0.0)
        self.mae_tracker.update_state(y, student_pred)

        return {
            "loss": self.loss_tracker.result(),
            "student_loss": self.student_loss_tracker.result(),
            "distill_loss": self.distill_loss_tracker.result(),
            "mae": self.mae_tracker.result(),
        }


# =========================
# EXPORT TFLITE
# =========================
def export_tflite_float32(model: tf.keras.Model, out_path: str) -> int:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = []
    tflite_model = converter.convert()
    with open(out_path, "wb") as f:
        f.write(tflite_model)
    return os.path.getsize(out_path)

def make_representative_dataset(X_samples: np.ndarray, batch_size: int = 1):
    X_samples = X_samples.astype(np.float32)

    def gen():
        for i in range(len(X_samples)):
            x = X_samples[i:i+1]
            yield [x]
    return gen

def export_tflite_ptq_int8(model: tf.keras.Model, Xtr_s: np.ndarray, out_path: str,
                           n_samples: int = 300) -> int:
    n = len(Xtr_s)
    if n == 0:
        raise ValueError("Xtr_s vazio - não dá pra calibrar PTQ.")

    n_use = min(n_samples, n)
    idx = np.linspace(0, n - 1, num=n_use, dtype=int)
    X_rep = Xtr_s[idx].astype(np.float32)

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = make_representative_dataset(X_rep, batch_size=REP_DATASET_BATCH)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()

    with open(out_path, "wb") as f:
        f.write(tflite_model)

    return os.path.getsize(out_path)

def extract_io_quant_params(tflite_path: str) -> dict:
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    in_details = interpreter.get_input_details()[0]
    out_details = interpreter.get_output_details()[0]

    in_scale, in_zero = in_details.get("quantization", (None, None))
    out_scale, out_zero = out_details.get("quantization", (None, None))

    info = {
        "input": {
            "name": in_details.get("name"),
            "shape": [int(x) for x in in_details.get("shape")],
            "dtype": str(in_details.get("dtype")),
            "quantization": {
                "scale": float(in_scale) if in_scale is not None else None,
                "zero_point": int(in_zero) if in_zero is not None else None,
            },
        },
        "output": {
            "name": out_details.get("name"),
            "shape": [int(x) for x in out_details.get("shape")],
            "dtype": str(out_details.get("dtype")),
            "quantization": {
                "scale": float(out_scale) if out_scale is not None else None,
                "zero_point": int(out_zero) if out_zero is not None else None,
            },
        },
    }
    return info


# =========================
# EXPORT HEADER .H
# =========================
def bytes_to_c_array(data: bytes, var_name: str = "g_model") -> str:
    hex_bytes = [f"0x{b:02x}" for b in data]
    lines = []
    lines.append("#pragma once")
    lines.append("#include <cstdint>")
    lines.append("")
    lines.append(f"// Model data: {len(data)} bytes")
    lines.append(f"alignas(16) const unsigned char {var_name}[] = {{")
    for i in range(0, len(hex_bytes), 12):
        chunk = ", ".join(hex_bytes[i:i+12])
        lines.append(f"  {chunk},")
    lines.append("};")
    lines.append(f"const unsigned int {var_name}_len = {len(data)};")
    lines.append("")
    return "\n".join(lines)

def export_model_h_from_tflite(tflite_path: str, out_h_path: str, var_name: str = "g_model"):
    with open(tflite_path, "rb") as f:
        data = f.read()
    header = bytes_to_c_array(data, var_name=var_name)
    with open(out_h_path, "w", encoding="utf-8") as f:
        f.write(header)


# =========================
# MAIN
# =========================
def main():
    set_seeds(SEED)

    # 1) Dados
    df = load_dataframe()
    X, y = build_sequences(df)
    Xtr, ytr, Xval, yval = split_tail(X, y, VAL_TAIL_FRAC)
    Xtr_s, Xval_s, scaler = scale_X_train_val(Xtr, Xval)

    # 2) Salva scaler
    scaler_info = {
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
        "var": scaler.var_.tolist(),
        "ctx": CTX,
        "n_features_flat": int(Xtr_s.reshape(len(Xtr_s), -1).shape[1]),
    }
    scaler_path = os.path.join(BASE_DIR, "scaler_standard.json")
    with open(scaler_path, "w") as f:
        json.dump(scaler_info, f, indent=2)

    in_features = Xtr_s.shape[-1]

    # =========================
    # 3) Treina teacher
    # =========================
    teacher = build_tinycnn(
        in_features=in_features,
        channels=HP_TEACHER["channels"],
        l2_weight=HP_TEACHER["l2_weight"],
        name="TinyCNN1D_Teacher"
    )
    teacher = compile_model(
        teacher,
        lr=HP_TEACHER["lr"],
        weight_decay=HP_TEACHER["weight_decay"]
    )

    teacher_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=int(HP_TEACHER["patience"]),
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(
            os.path.join(BASE_DIR, "teacher_train_log.csv"),
            append=False
        ),
    ]

    teacher.fit(
        Xtr_s, ytr,
        validation_data=(Xval_s, yval),
        epochs=int(HP_TEACHER["max_epochs"]),
        batch_size=int(HP_TEACHER["batch_size"]),
        shuffle=False,
        verbose=1,
        callbacks=teacher_callbacks,
    )

    teacher_preds = teacher.predict(Xval_s, batch_size=int(HP_TEACHER["batch_size"]), verbose=0).reshape(-1)
    teacher_rmse, teacher_mae = rmse_mae(yval, teacher_preds)

    # =========================
    # 4) Treina student com KD
    # =========================
    student = build_tinycnn(
        in_features=in_features,
        channels=HP_STUDENT["channels"],
        l2_weight=HP_STUDENT["l2_weight"],
        name="TinyCNN1D_Student_KD"
    )

    try:
        student_opt = tf.keras.optimizers.AdamW(
            learning_rate=HP_STUDENT["lr"],
            weight_decay=HP_STUDENT["weight_decay"]
        )
    except Exception:
        student_opt = tf.keras.optimizers.Adam(
            learning_rate=HP_STUDENT["lr"]
        )

    distiller = RegressionDistiller(
        student=student,
        teacher=teacher,
        alpha=KD_CONFIG["alpha"],
        temperature=KD_CONFIG["temperature"],
    )

    distiller.compile(
        optimizer=student_opt,
        student_loss_fn=tf.keras.losses.MeanSquaredError(),
        distillation_loss_fn=tf.keras.losses.MeanSquaredError(),
    )

    kd_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=int(HP_STUDENT["patience"]),
            restore_best_weights=True,
            verbose=1,
        ),
        SimpleCSVLogger(os.path.join(BASE_DIR, "student_kd_train_log.csv")),
    ]

    distiller.fit(
        Xtr_s, ytr,
        validation_data=(Xval_s, yval),
        epochs=int(HP_STUDENT["max_epochs"]),
        batch_size=int(HP_STUDENT["batch_size"]),
        shuffle=False,
        verbose=1,
        callbacks=kd_callbacks,
    )

    student_preds = student.predict(Xval_s, batch_size=int(HP_STUDENT["batch_size"]), verbose=0).reshape(-1)
    student_rmse, student_mae = rmse_mae(yval, student_preds)

    # =========================
    # 5) Export teacher float32
    # =========================
    teacher_tflite_path = os.path.join(BASE_DIR, "tinycnn_teacher_float32.tflite")
    teacher_tflite_bytes = export_tflite_float32(teacher, teacher_tflite_path)

    teacher_h_path = os.path.join(BASE_DIR, "model_teacher.h")
    export_model_h_from_tflite(teacher_tflite_path, teacher_h_path, var_name="g_tinycnn_teacher")

    # =========================
    # 6) Export student KD float32
    # =========================
    kd_tflite_path = os.path.join(BASE_DIR, "tinycnn_kd_float32.tflite")
    kd_tflite_bytes = export_tflite_float32(student, kd_tflite_path)

    kd_h_path = os.path.join(BASE_DIR, "model_kd.h")
    export_model_h_from_tflite(kd_tflite_path, kd_h_path, var_name="g_tinycnn_kd")

    # =========================
    # 7) Export student KD PTQ int8
    # =========================
    kd_int8_tflite_path = os.path.join(BASE_DIR, "tinycnn_kd_ptq_int8.tflite")
    kd_int8_tflite_bytes = export_tflite_ptq_int8(
        student,
        Xtr_s=Xtr_s,
        out_path=kd_int8_tflite_path,
        n_samples=REP_DATASET_SAMPLES,
    )

    kd_int8_h_path = os.path.join(BASE_DIR, "model_kd_ptq_int8.h")
    export_model_h_from_tflite(kd_int8_tflite_path, kd_int8_h_path, var_name="g_tinycnn_kd_ptq_int8")

    quant_params = extract_io_quant_params(kd_int8_tflite_path)
    quant_params_path = os.path.join(BASE_DIR, "quant_params_kd_ptq_int8.json")
    with open(quant_params_path, "w") as f:
        json.dump(quant_params, f, indent=2)

    # =========================
    # 8) Metrics
    # =========================
    metrics = {
        "variant_teacher": "tinycnn_teacher_float32",
        "variant_student_kd_float32": "tinycnn_kd_float32",
        "variant_student_kd_ptq_int8": "tinycnn_kd_ptq_int8",

        "teacher_metrics_val": {
            "RMSE": teacher_rmse,
            "MAE": teacher_mae,
            "n_params": int(teacher.count_params()),
        },

        "student_kd_metrics_val": {
            "RMSE": student_rmse,
            "MAE": student_mae,
            "n_params": int(student.count_params()),
        },

        "teacher_tflite_float32": {
            "path": teacher_tflite_path,
            "bytes": int(teacher_tflite_bytes),
            "model_h": teacher_h_path,
            "c_var": "g_tinycnn_teacher",
        },

        "student_kd_tflite_float32": {
            "path": kd_tflite_path,
            "bytes": int(kd_tflite_bytes),
            "model_h": kd_h_path,
            "c_var": "g_tinycnn_kd",
        },

        "student_kd_tflite_ptq_int8": {
            "path": kd_int8_tflite_path,
            "bytes": int(kd_int8_tflite_bytes),
            "model_h": kd_int8_h_path,
            "c_var": "g_tinycnn_kd_ptq_int8",
            "quant_params_path": quant_params_path,
        },

        "input_shape": [None, CTX, int(in_features)],
        "teacher_hyperparams": HP_TEACHER,
        "student_hyperparams": HP_STUDENT,
        "kd_config": KD_CONFIG,

    }

    metrics_path = os.path.join(BASE_DIR, "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    print("\n[OK] Treino finalizado.")
    print(f"[OK] Teacher -> RMSE_val = {teacher_rmse:.6f} | MAE_val = {teacher_mae:.6f}")
    print(f"[OK] Student KD -> RMSE_val = {student_rmse:.6f} | MAE_val = {student_mae:.6f}")
    print(f"[OK] Teacher float32: {teacher_tflite_path} ({teacher_tflite_bytes} bytes)")
    print(f"[OK] Student KD float32: {kd_tflite_path} ({kd_tflite_bytes} bytes)")
    print(f"[OK] Student KD PTQ int8: {kd_int8_tflite_path} ({kd_int8_tflite_bytes} bytes)")
    print(f"[OK] Quant params KD int8: {quant_params_path}")
    print(f"[OK] Metrics: {metrics_path}")
    print(f"[OK] Scaler: {scaler_path}")


if __name__ == "__main__":
    main()